# Sales ledger checkpoint demo (SQL-centric version)

Four short experiments, same visual language throughout:
**Receipt = JSON commit. Summary = Checkpoint. Today's Table = Latest snapshot. Old Version = Time travel.**

- **A — Healthy ledger**: how Delta normally works.
- **B — Delete an old receipt** (before a later summary): does today's total still work? Does that old day's exact report still work?
- **C — Delete a receipt after the summary, but not the newest one**: can Delta still cross that gap?
- **D — Delete the newest receipt**: what happens on the very next sale?

**This notebook uses SQL commands wherever possible.** Python is only used for:
- File system operations (deleting Delta log files)
- Finding checkpoint versions (SQL can't list files)
- Helper functions for testing

Every experiment ends in the same three-line scoreboard:
`LATEST READ` / `TIME TRAVEL` / `NEW WRITE` — pass or fail, one line each.

### Step 0 — Reset
Clear all demo paths for a fresh start.

In [0]:
BASE = "/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo"
BEFORE_PATH = "/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_before_cp"
AFTER_PATH  = "/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_after_cp"
LATEST_PATH = "/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_latest_delete"

for p in [BASE, BEFORE_PATH, AFTER_PATH, LATEST_PATH]:
    try:
        dbutils.fs.rm(p, recurse=True)
        print(f"[RESET] Cleared: {p}")
    except:
        pass

[RESET] Cleared: /Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo
[RESET] Cleared: /Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_before_cp
[RESET] Cleared: /Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_after_cp
[RESET] Cleared: /Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_latest_delete


### Step 1 — Create the sales table (SQL)
`delta.checkpointInterval` is set to 3, so a checkpoint gets written right after the 3rd sale.

In [0]:
%sql
CREATE TABLE delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo` (
  sale_id INT,
  item STRING,
  amount DOUBLE
)
USING DELTA
TBLPROPERTIES (
  'delta.checkpointInterval' = '3',
  'delta.autoOptimize.autoCompact' = 'false',
  'delta.autoOptimize.optimizeWrite' = 'false'
);

-- Verify empty table
SELECT '[CREATE] Table created (Version 0, empty)' as status;
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo`;

sale_id,item,amount


### Step 2 — Ring up 6 sales, one commit at a time (SQL)
Each INSERT is its own version. We'll check the running total after each one.

In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo` VALUES (1, 'Coffee', 5.0);
SELECT '[Sale 1] Coffee, $5.0' as sale, SUM(amount) as running_total FROM delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo`;

sale,running_total
"[Sale 1] Coffee, .0",5.0


In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo` VALUES (2, 'Sandwich', 8.0);
SELECT '[Sale 2] Sandwich, $8.0' as sale, SUM(amount) as running_total FROM delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo`;

sale,running_total
"[Sale 2] Sandwich, .0",13.0


In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo` VALUES (3, 'Coffee', 5.0);
SELECT '[Sale 3] Coffee, $5.0 → CHECKPOINT CREATED HERE' as sale, SUM(amount) as running_total FROM delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo`;

sale,running_total
"[Sale 3] Coffee, .0 → CHECKPOINT CREATED HERE",18.0


In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo` VALUES (4, 'Cake', 6.0);
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo` VALUES (5, 'Tea', 4.0);
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo` VALUES (6, 'Coffee', 5.0);
SELECT '[Sales 4-6] All sales complete' as status, SUM(amount) as final_total FROM delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo`;

status,final_total
[Sales 4-6] All sales complete,33.0


### Step 3 — Check Delta history (SQL) and find checkpoint (Python helper)

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_demo`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-08-10T19:41:51.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(732281355111939),b62b169c-566e-4ad2-8780-b77427ba181f,0810-182949-ox2qz7mh-v2n,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1223)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-08-10T19:41:50.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(732281355111939),d08438f2-f082-4cb3-a667-3857518eec32,0810-182949-ox2qz7mh-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1208)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-10T19:41:49.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(732281355111939),787c6bc8-832d-41b9-84c1-0fe4985383df,0810-182949-ox2qz7mh-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1213)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-10T19:41:45.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(732281355111939),f60039bc-4d77-4194-9234-908af795ec38,0810-182949-ox2qz7mh-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1222)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-10T19:41:42.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(732281355111939),d52f50bc-4fa3-4287-8f22-af0afa4a8786,0810-182949-ox2qz7mh-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1233)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-10T19:41:40.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(732281355111939),6edcb031-2973-43ec-8017-9517bda34f6b,0810-182949-ox2qz7mh-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1223)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-10T19:41:37.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.checkpointInterval"":""3"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(732281355111939),822fdf2f-020c-4e50-8be0-938644b5754a,0810-182949-ox2qz7mh-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
import re

def find_checkpoint_version(path):
    files = dbutils.fs.ls(f"{path}/_delta_log/")
    versions = []
    for f in files:
        m = re.match(r"^(\d+)\.checkpoint(\.\d+\.\d+)?\.parquet$", f.name)
        if m:
            versions.append(int(m.group(1)))
    return max(versions) if versions else None

checkpoint_v = find_checkpoint_version(BASE)
all_json = sorted([f.name for f in dbutils.fs.ls(f"{BASE}/_delta_log/") if f.name.endswith(".json")])
print("JSON commit files:", all_json)
print("Checkpoint found at version:", checkpoint_v)
if checkpoint_v is None:
    print("No checkpoint yet — re-run this cell or add one more sale.")

JSON commit files: ['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json', '00000000000000000003.json', '00000000000000000004.json', '00000000000000000005.json', '00000000000000000006.json']
Checkpoint found at version: 6


### Step 4 — Build three twin ledgers for Experiments B, C, D
Built fresh with the same structure. Python loops the SQL commands.

In [0]:
def build_ledger_sql(path, sale_list):
    spark.sql(f"""
        CREATE TABLE delta.`{path}` (
          sale_id INT, item STRING, amount DOUBLE
        )
        USING DELTA
        TBLPROPERTIES (
          'delta.checkpointInterval' = '3',
          'delta.autoOptimize.autoCompact' = 'false',
          'delta.autoOptimize.optimizeWrite' = 'false'
        )
    """)
    for sid, item, amt in sale_list:
        spark.sql(f"INSERT INTO delta.`{path}` VALUES ({sid}, '{item}', {amt})")
    cp = find_checkpoint_version(path)
    v = spark.sql(f"DESCRIBE HISTORY delta.`{path}`").agg({"version": "max"}).collect()[0][0]
    print(f"[BUILD] {path.split('/')[-1]:35s} -> v{v}, checkpoint at {cp}")

sales = [(1, "Coffee", 5.0), (2, "Sandwich", 8.0), (3, "Coffee", 5.0),
         (4, "Cake", 6.0), (5, "Tea", 4.0), (6, "Coffee", 5.0)]

build_ledger_sql(BEFORE_PATH, sales)
build_ledger_sql(AFTER_PATH, sales + [(7, "Muffin", 4.5), (8, "Bagel", 4.0)])
build_ledger_sql(LATEST_PATH, sales)

[BUILD] sql_sales_ledger_before_cp          -> v6, checkpoint at 6
[BUILD] sql_sales_ledger_after_cp           -> v8, checkpoint at 6
[BUILD] sql_sales_ledger_latest_delete      -> v6, checkpoint at 6


### Shared testing framework (Python helpers)
These functions test read/write operations and format results consistently.

In [0]:
def short_reason(e):
    text = str(e)
    for stop in ["\n\nJVM stacktrace", "\nJVM stacktrace", "\n\tat "]:
        if stop in text:
            text = text.split(stop)[0]
            break
    text = text.strip()
    return text if len(text) <= 160 else text[:157] + "..."

def try_read_sql(path, version_as_of=None):
    try:
        df = spark.read.format("delta")
        if version_as_of is not None:
            df = df.option("versionAsOf", version_as_of)
        df.load(path).count()
        return True, None
    except Exception as e:
        return False, short_reason(e)

def try_write_sql(path, sid, item, amt):
    try:
        spark.sql(f"INSERT INTO delta.`{path}` VALUES ({sid}, '{item}', {amt})")
        v = spark.sql(f"DESCRIBE HISTORY delta.`{path}`").agg({"version": "max"}).collect()[0][0]
        return True, v
    except Exception as e:
        return False, short_reason(e)

def scoreboard(title, rows):
    print(f"\n{title}")
    print("-" * len(title))
    for label, passed, detail in rows:
        mark = "PASS" if passed else "FAIL"
        line = f"  {label:<28s} {mark}"
        if detail:
            line += f"   ({detail})"
        print(line)

### Experiment B — Delete an OLD receipt (before the summary)
Delete Sale #2's commit (before checkpoint at version 3).

In [0]:
target_before = 2
dbutils.fs.rm(f"{BEFORE_PATH}/_delta_log/{target_before:020d}.json")

latest_ok, latest_detail = try_read_sql(BEFORE_PATH)
vao_ok, vao_detail = try_read_sql(BEFORE_PATH, version_as_of=target_before)
write_ok, write_detail = try_write_sql(BEFORE_PATH, 99, "Test Item", 1.0)

scoreboard("EXPERIMENT B -- Delete Sale #2 (before the summary)", [
    ("Can I read LATEST?", latest_ok, latest_detail),
    (f"Can I TIME TRAVEL to v{target_before}?", vao_ok, vao_detail),
    ("Can I make a NEW WRITE?", write_ok, f"landed at v{write_detail}" if write_ok else write_detail),
])

before_latest_read, before_vao, before_new_write = latest_ok, vao_ok, write_ok


EXPERIMENT B -- Delete Sale #2 (before the summary)
---------------------------------------------------
  Can I read LATEST?           PASS
  Can I TIME TRAVEL to v2?     FAIL   (requirement failed: Did not get the last delta file version: 2 to compute Snapshot)
  Can I make a NEW WRITE?      PASS   (landed at v7)


### Experiment C — Delete a receipt AFTER the summary (not the newest)
Delete Sale #7's commit. Sale #8 still exists, proving a gap in the chain.

In [0]:
target_after = 7
dbutils.fs.rm(f"{AFTER_PATH}/_delta_log/{target_after:020d}.json")

latest_ok, latest_detail = try_read_sql(AFTER_PATH)
vao7_ok, vao7_detail = try_read_sql(AFTER_PATH, version_as_of=7)
vao8_ok, vao8_detail = try_read_sql(AFTER_PATH, version_as_of=8)
write_ok, write_detail = try_write_sql(AFTER_PATH, 9, "Croissant", 5.5)

scoreboard("EXPERIMENT C -- Delete Sale #7, Sale #8 survives", [
    ("Can I read LATEST?", latest_ok, latest_detail),
    ("Can I TIME TRAVEL to v7?", vao7_ok, vao7_detail),
    ("Can I TIME TRAVEL to v8?", vao8_ok, vao8_detail),
    ("Can I make a NEW WRITE?", write_ok, f"landed at v{write_detail}" if write_ok else write_detail),
])

after_latest_read, after_vao_7, after_vao_8, after_new_write = latest_ok, vao7_ok, vao8_ok, write_ok


EXPERIMENT C -- Delete Sale #7, Sale #8 survives
------------------------------------------------
  Can I read LATEST?           FAIL   ((com.databricks.sql.transaction.tahoe.DeltaFileNotFoundException) [DELTA_TRUNCATED_TRANSACTION_LOG] dbfs:/Volumes/workspace/delta_demo/demo_files/sql_sales_l...)
  Can I TIME TRAVEL to v7?     FAIL   ((com.databricks.sql.transaction.tahoe.DeltaFileNotFoundException) [DELTA_TRUNCATED_TRANSACTION_LOG] dbfs:/Volumes/workspace/delta_demo/demo_files/sql_sales_l...)
  Can I TIME TRAVEL to v8?     FAIL   ((com.databricks.sql.transaction.tahoe.DeltaFileNotFoundException) [DELTA_TRUNCATED_TRANSACTION_LOG] dbfs:/Volumes/workspace/delta_demo/demo_files/sql_sales_l...)
  Can I make a NEW WRITE?      FAIL   ((com.databricks.sql.transaction.tahoe.DeltaFileNotFoundException) [DELTA_TRUNCATED_TRANSACTION_LOG] dbfs:/Volumes/workspace/delta_demo/demo_files/sql_sales_l...)


### Experiment D — Delete the NEWEST receipt
Delete Sale #6 (the newest) and see if the next write reuses version 6.

In [0]:
latest_before_delete = spark.sql(f"DESCRIBE HISTORY delta.`{LATEST_PATH}`").agg({"version": "max"}).collect()[0][0]
target_latest = latest_before_delete
dbutils.fs.rm(f"{LATEST_PATH}/_delta_log/{target_latest:020d}.json")

write_ok, new_version = try_write_sql(LATEST_PATH, 7, "Muffin", 4.5)

print(f"\nEXPERIMENT D -- Delete Sale #{target_latest} (the newest), then ring up a new sale")
print("-" * 60)
print(f"  Version deleted        : {target_latest}")
if write_ok:
    print(f"  Next sale landed at    : version {new_version}")
    if new_version == target_latest:
        print(f"  -> Delta REUSED version {target_latest}. It never saw Sale #{target_latest} commit as existing.")
    else:
        print(f"  -> Delta skipped past version {target_latest} to {new_version}.")
else:
    print(f"  New write FAILED: {new_version}")

latest_delete_reused = write_ok and (new_version == target_latest)


EXPERIMENT D -- Delete Sale #6 (the newest), then ring up a new sale
------------------------------------------------------------
  Version deleted        : 6
  New write FAILED: (java.lang.IllegalStateException) Could not find any delta files for version 6


### Step 5 — Validate all at once
Auto-generated summary of all experiments.

In [0]:
print("EXPERIMENT B -- delete BEFORE the summary (Sale #2)")
print(f"  Latest read : {'PASS' if before_latest_read else 'FAIL'}   Time travel : {'PASS' if before_vao else 'FAIL'}   New write : {'PASS' if before_new_write else 'FAIL'}")
print()
print("EXPERIMENT C -- delete AFTER the summary, not newest (Sale #7, #8 survives)")
print(f"  Latest read : {'PASS' if after_latest_read else 'FAIL'}   Time travel v7 : {'PASS' if after_vao_7 else 'FAIL'}   Time travel v8 : {'PASS' if after_vao_8 else 'FAIL'}   New write : {'PASS' if after_new_write else 'FAIL'}")
print()
print("EXPERIMENT D -- delete the NEWEST receipt")
print(f"  Next write reused the deleted version number : {'YES' if latest_delete_reused else 'NO'}")
print()
print("=" * 60)
print("CHECKPOINT != BACKUP")
print("A checkpoint helps Delta reconstruct table state efficiently.")
print("It does NOT guarantee every old version survives manual")
print("deletion of transaction logs.")
print("=" * 60)

EXPERIMENT B -- delete BEFORE the summary (Sale #2)
  Latest read : PASS   Time travel : FAIL   New write : PASS

EXPERIMENT C -- delete AFTER the summary, not newest (Sale #7, #8 survives)
  Latest read : FAIL   Time travel v7 : FAIL   Time travel v8 : FAIL   New write : FAIL

EXPERIMENT D -- delete the NEWEST receipt
  Next write reused the deleted version number : NO

CHECKPOINT != BACKUP
A checkpoint helps Delta reconstruct table state efficiently.
It does NOT guarantee every old version survives manual
deletion of transaction logs.


### Bonus — Fresh-session verification for Experiment D
Runs after the summary. `%restart_python` wipes the session to re-confirm Experiment D with zero cached reads.

In [0]:
%restart_python

### Fresh session verification

**What happened in Experiment D:**
* We deleted version 6 (the newest receipt).
* Python was restarted to clear all cached metadata.

**What we're about to test:**
* Can we insert a new sale into this table?
* Will Delta reuse version 6, or skip to version 7?

**Note:** We CANNOT read or describe this table right now — even `DESCRIBE HISTORY` fails because the latest commit file (version 6) is missing. This is exactly what Experiment D showed: deleting the newest receipt breaks **everything** until a new write repairs the chain.

In [0]:
LATEST_PATH = "/Volumes/workspace/delta_demo/demo_files/sql_sales_ledger_latest_delete"

try:
    spark.sql(f"INSERT INTO delta.`{LATEST_PATH}` VALUES (7, 'Muffin', 4.5)")
    v = spark.sql(f"DESCRIBE HISTORY delta.`{LATEST_PATH}`").agg({"version": "max"}).collect()[0][0]
    print(f"FRESH SESSION -- New write: SUCCESS, landed at version {v}")
except Exception as e:
    print(f"FRESH SESSION -- New write: FAILED")
    print(f"Reason: {str(e).split(chr(10))[0]}")

FRESH SESSION -- New write: FAILED
Reason: (java.lang.IllegalStateException) Could not find any delta files for version 6
